# DTL: export satellite image to asset

Save satellite image stack to an asset

In [4]:
# !python -m pip install .. --quiet

import ee 

ee.Authenticate() 
ee.Initialize(project='epistem2')

In [5]:
VERSION = 'v7'
REGION = 'Sumatra'
region_lower = REGION.lower()


In [6]:
# 1. Satellite imagery
# AOI definition

# import region from Hadi's assets
# region_name = "Sumatera"
# regions_fc = ee.FeatureCollection("users/hadicu06/IIASA/RESTORE/vector_datasets/classification_regions")
# aoi = regions_fc.filter(ee.Filter.eq('region_name', region_name)).geometry()

aoi = ee.FeatureCollection(f"projects/epistem2/assets/AOI_{REGION}")


import geemap
from luma_ge.data_acquisition import Reflectance_Data, final_Image

#========== FIRST RETRIVE THE MULTISPECTRAL BAND===========
#Intialize the relfectance class data function
optical_reflectance = Reflectance_Data()
#Initialize the final image class for composite creation
composite = final_Image() #NEW FEATURE ADDED HERE
#define the start and end date for imagery collection
start = '2021-01-01'
end = '2021-12-31'
from luma_ge.data_acquisition import Reflectance_Data, final_Image

optical_reflectance = Reflectance_Data()

composite = final_Image() #NEW FEATURE ADDED HERE
#define the start and end date for imagery collection
start = '2021-01-01'
end = '2021-12-31'

landsat_data, meta = optical_reflectance.get_optical_data(
    aoi, start, end, optical_data='L8_SR', 
    cloud_cover=50,
    compute_detailed_stats=False,
    scene_limit=500
)

stacked_landsat = landsat_data.mosaic().clip(aoi)

# stacked_landsat = composite.get_temporal_composite(
#     landsat_data, aoi, 
#     coverage_scale=100 ,
#     calculate_coverage=True
# )

2026-09-08 11:01:17,593 - luma_ge.ee_config - INFO - Earth Engine initialized successfully
2026-09-08 11:01:17,595 - Reflectance_Data - INFO - ReflectanceData initialized.
2026-09-08 11:01:17,596 - final_Image - INFO - final_Image creation initialized.
2026-09-08 11:01:17,598 - Reflectance_Data - INFO - ReflectanceData initialized.
2026-09-08 11:01:17,599 - final_Image - INFO - final_Image creation initialized.
2026-09-08 11:01:17,600 - Reflectance_Data - INFO - Starting data fetch for Landsat 8 Operational Land Imager Surface Reflectance
2026-09-08 11:01:17,601 - Reflectance_Data - INFO - Date range: 2021-01-01 to 2021-12-31
2026-09-08 11:01:17,602 - Reflectance_Data - INFO - Cloud cover threshold: 50%
2026-09-08 11:01:17,605 - Reflectance_Data - INFO - detailed statistics will not be computed
2026-09-08 11:01:17,606 - Reflectance_Stats - INFO - Reflectance Stats initialized.
2026-09-08 11:01:17,607 - Reflectance_Data - INFO - Filtered collection created (use compute_detailed_stats=Tr

In [ ]:
# l8_sr_visparam = {'min': 0,'max': 0.4,'gamma': [0.95, 1.1, 1],'bands':['BLUE', 'RED', 'GREEN']}

# Map = geemap.Map()

# Map.addLayer(stacked_landsat, l8_sr_visparam, 'stacked_landsat')

# Map

## Add predictor to satellite image stack

In [7]:
from luma_ge.predictor import SpectralCalculator

spectral_calc = SpectralCalculator()

indices_to_compute = ['NDVI', 'MNDWI', 'NDBI']
spectral_indices = spectral_calc.calculate_indices_with_collection(
    collection=landsat_data,
    aoi=aoi, # type: ignore
    index_list=indices_to_compute,
    reducer_method='mean'
)

final_stack = stacked_landsat.addBands(spectral_indices).toFloat()

2026-09-08 11:01:26,954 - luma_ge.predictor - INFO - SpectralCalculator initialized
2026-09-08 11:01:26,954 - luma_ge.predictor - INFO - Calculating spectral indices using map and mean reducer on collection
2026-09-08 11:01:26,955 - luma_ge.predictor - INFO - Using default coefficients where required
2026-09-08 11:01:26,957 - luma_ge.predictor - INFO - Mapping index calculation over collection...
2026-09-08 11:01:27,328 - luma_ge.predictor - INFO - Applying mean reducer to collection...
2026-09-08 11:01:44,545 - luma_ge.predictor - INFO - Successfully calculated 3 spectral indices using map and mean reducer


In [8]:
task = ee.batch.Export.image.toAsset(
    image=final_stack,
    description=f'stacked_landsat_pred_2021_{region_lower}_{VERSION}',
    assetId=f'projects/epistem2/assets/stacked_landsat_pred_2021_{region_lower}_{VERSION}',
    region=aoi.geometry(),
    scale=100,
    maxPixels=1e13
)

task.start()